In [4]:
# 1

%pip install supabase python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
#2 
from supabase import create_client
from dotenv import load_dotenv

print ("도구 준비 완료")

도구 준비 완료


In [2]:
#3

import os 
from pathlib import Path


url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

supabase = create_client(url, key)


In [41]:
#4 판매자 계정 생성

load_dotenv(Path.cwd() / ".env", override=True)

seller_email = os.getenv("SELLER_EMAIL")
seller_password = os.getenv("SELLER_PASSWORD")

response = supabase.auth.sign_up({
    "email" : seller_email,
    "password" : seller_password
})

print("판매자 계정 생성 요청 완료")
print("사용자 ID:", response.user.id)

AuthApiError: User already registered

In [48]:
#5 로그인 단계

response = supabase.auth.sign_in_with_password({
    "email" : seller_email,
    "password" : seller_password   
})

In [49]:
#6 현재 로그인한 사용자 확인
# -- 지금 파이썬이 누구 권한으로 supabase에 접근하고 있는지 확인하는 단계

current_user_response = supabase.auth.get_user()
current_user = current_user_response.user 

current_user

User(id='f6f9dc8e-d48a-4fec-bc98-09ea3773859f', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'seller01@example.com', 'email_verified': True, 'phone_verified': False, 'sub': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='seller01@example.com', phone='', created_at=datetime.datetime(2026, 9, 13, 10, 20, 5, 801817, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 13, 10, 20, 5, 844543, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 13, 10, 20, 5, 844543, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 10, 5, 10, 57, 23, 453235, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 10, 5, 10, 57, 23, 467404, tzinfo=TzInfo(0)), identities=[UserIdentity(id='f6f9dc8e-d48a-4fec-bc98-09ea3773859f', identity_i

In [50]:
#7 .py 만들기
#8 .py파일 가져와서 로그인한 판매자 정보를 클래스방식으로 가져온다

from user_details import UserService

user_service = UserService(supabase)
#수퍼베이스 리모컨을 사용자 담당클래스에 전달

auth_user = user_service.get_auth_user()

print("로그인한 사용자 정보:", auth_user.id)
print("로그인한 사용자 이메일:", auth_user.email)


로그인한 사용자 정보: f6f9dc8e-d48a-4fec-bc98-09ea3773859f
로그인한 사용자 이메일: seller01@example.com


In [51]:
#9 .py파일에 프로필 생성기능 추가
#10 판매자 프로필 생성

seller_profile = user_service.create_profile(
    role='seller',
    mobile = "010-1111-2222",
    address = "Seoul"
)

APIError: {'message': 'duplicate key value violates unique constraint "user_details_pkey"', 'code': '23505', 'hint': None, 'details': 'Key (id)=(f6f9dc8e-d48a-4fec-bc98-09ea3773859f) already exists.'}

In [ ]:
#11 프로필 조회생성기능 추가
#12 프로필 조회

In [12]:
import importlib
import user_details  # user_details 모듈 불러오기

# 1. user_details 모듈을 강제로 최신 상태로 새로고침
importlib.reload(user_details)

# 2. 새로고침된 모듈에서 UserService 클래스 가져오기
from user_details import UserService

user_service = UserService(supabase)

seller_profile = user_service.get_profile()

print(seller_profile)

{'id': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f', 'role': 'seller', 'mobile': '010-1111-2222', 'email': None, 'address': 'Seoul', 'city': None, 'first_name': None, 'last_name': None, 'created_at': '2026-09-13T11:52:46.793732+00:00', 'modified_at': '2026-09-13T11:52:46.793732+00:00', 'deleted_at': None}


In [13]:
auth_user = user_service.get_auth_user()

all_profiles = (
    supabase.table("user_details")
    .select("id, role, email")
    .execute()
)

print("현재 로그인한 Auth ID:", auth_user.id)
print("DB에 있는 프로필 목록:", all_profiles.data)

현재 로그인한 Auth ID: f6f9dc8e-d48a-4fec-bc98-09ea3773859f
DB에 있는 프로필 목록: [{'id': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f', 'role': 'seller', 'email': None}]


--사용자 파트 끝--
Auth 계정 생성       ✅
판매자 로그인        ✅
user_details 프로필  ✅
프로필 조회          ✅

In [47]:
#13 product_service.py생성
#14 판매자인지 확인

from product_service import ProductService

product_service = ProductService(supabase)

print(product_service.is_seller())

AttributeError: 'NoneType' object has no attribute 'user'

In [28]:
import importlib
import product_service

importlib.reload(product_service)

ProductService = product_service.ProductService

product_service = ProductService(supabase)

print("상품 담당 클래스 준비 완료")

상품 담당 클래스 준비 완료


In [46]:
print("현재 사용자는 판매자인가?", product_service.is_seller())

AttributeError: 'NoneType' object has no attribute 'user'

In [32]:

#15 프로덕트 생성 
created_product = product_service.create_product(
    name="갈비만두",
    price=12000,
    description="복습용 상품"
)

In [37]:
from pprint import pprint
pprint(created_product)

pprint(created_product)

[{'created_at': '2026-10-05T10:46:45.267472+00:00',
  'deleted_at': None,
  'description': '복습용 상품',
  'id': '32da4534-4563-4bcb-828d-8db449133762',
  'modified_at': '2026-10-05T10:46:45.267472+00:00',
  'name': '갈비만두',
  'price': 12000,
  'seller_id': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f',
  'stock_quantity': 100}]
[{'created_at': '2026-10-05T10:46:45.267472+00:00',
  'deleted_at': None,
  'description': '복습용 상품',
  'id': '32da4534-4563-4bcb-828d-8db449133762',
  'modified_at': '2026-10-05T10:46:45.267472+00:00',
  'name': '갈비만두',
  'price': 12000,
  'seller_id': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f',
  'stock_quantity': 100}]


In [44]:
#16 프로덕트 조회
product_id = created_product[0]["id"]

print(product_id)

32da4534-4563-4bcb-828d-8db449133762


In [45]:
#17 프로덕트 수정
updated_product = product_service.update_product(
    product_id = product_id,
    price= 15000
)

print(updated_product)

AttributeError: 'NoneType' object has no attribute 'user'

In [ ]:
#18 프로덕트 삭제

deleted_product = product_service.delete_product(product_id)

print(deleted_product)



[{'id': '3fa9c4b1-670c-4027-9a8d-8da4c01df364', 'seller_id': 'f6f9dc8e-d48a-4fec-bc98-09ea3773859f', 'name': '갈비만두', 'price': 15000, 'description': '복습용 상품', 'created_at': '2026-09-13T13:33:32.88497+00:00', 'modified_at': '2026-09-13T13:33:32.88497+00:00', 'deleted_at': '2026-09-13T23:07:43.164522+00:00'}]


In [28]:
#19 .env파일에 구매자 계정 추가
#20 구매자 테스트 계정 생성

load_dotenv(Path.cwd() / ".env", override=True)

buyer_email = os.getenv ("BUYER_EMAIL")
buyer_password = os.getenv("BUYER_PASSWORD")

response = supabase.auth.sign_up({
    "email" : buyer_email,
    "password": buyer_password
})

print("구매자 계정 생성 요청 완료")
print("구매자 ID:", response.user.id)

구매자 계정 생성 요청 완료
구매자 ID: 8c9161e5-578e-4000-81fa-560af32eac56


In [29]:
response = supabase.auth.sign_in_with_password({
    "email": buyer_email,
    "password": buyer_password
})
print("구매자 로그인 완료")
print("현재 로그인한 사용자 정보:", response.user.id)

구매자 로그인 완료
현재 로그인한 사용자 정보: 8c9161e5-578e-4000-81fa-560af32eac56


In [30]:
buyer_service = UserService(supabase)

buyer_propfile = buyer_service.create_profile(
    role = "buyer",
    mobile = "010-3333-4444",
    address = "Seoul"
)

print(buyer_propfile)

[{'id': '8c9161e5-578e-4000-81fa-560af32eac56', 'role': 'buyer', 'mobile': '010-3333-4444', 'email': None, 'address': 'Seoul', 'city': None, 'first_name': None, 'last_name': None, 'created_at': '2026-09-13T14:17:30.46352+00:00', 'modified_at': '2026-09-13T14:17:30.46352+00:00', 'deleted_at': None}]


In [31]:
#21 orders, order_items 테이블 생성
#21 order_service.py 생성
#22 바이어 확인

import importlib
import order_service as osvc

osvc = importlib.reload(osvc)

shop_orders = osvc.OrderService(supabase)

print(shop_orders.is_buyer())

True


In [37]:
import importlib
import order_service

order_service = importlib.reload(order_service)

shop_orders = order_service.OrderService(supabase)

print(shop_orders.is_buyer())

True


In [38]:
# 주문가능한 상품 조회

products_response = (
    supabase.table("products")
    .select("id, name, price")
    .is_("deleted_at", "null")
    .execute()
)

available_products = products_response.data

print(available_products)

[{'id': '74c19d94-5ece-4be6-a01d-853d391c38fc', 'name': '김치만두', 'price': 10000}]


In [40]:

result = shop_orders.create_order(
    product_id='74c19d94-5ece-4be6-a01d-853d391c38fc',
    quantity = 2
)

print("주문 정보:", result["order"])
print("주문 상품 정보:", result["order_item"])

주문 정보: {'id': 'f9e1e26f-0d16-4e16-b1b0-1e67044e70c6', 'buyer_id': '8c9161e5-578e-4000-81fa-560af32eac56', 'status': 'pending', 'total_amount': 20000, 'created_at': '2026-09-13T14:59:25.42164+00:00', 'modified_at': '2026-09-13T14:59:25.42164+00:00'}
주문 상품 정보: {'id': '8c1aa4e4-5f11-4b35-8403-72d6ef418eb1', 'order_id': 'f9e1e26f-0d16-4e16-b1b0-1e67044e70c6', 'product_id': '74c19d94-5ece-4be6-a01d-853d391c38fc', 'quantity': 2, 'unit_price': 10000, 'created_at': '2026-09-13T14:59:25.445956+00:00'}


In [3]:
#구매자로 로그인

buyer_email = os.getenv("BUYER_EMAIL")
buyer_password = os.getenv("BUYER_PASSWORD")

supabase.auth.sign_in_with_password({
    "email": buyer_email,
    "password": buyer_password
})

current_user = supabase.auth.get_user().user

print("구매자 로그인 완료")
print("현재 사용자 ID:", current_user.id)

구매자 로그인 완료
현재 사용자 ID: 8c9161e5-578e-4000-81fa-560af32eac56


In [6]:
#get_my_orders

my_orders = shop_orders.get_my_orders()

print("내 주문 개수:", len(my_orders))

for order in my_orders:
    print("-" * 30)
    print("주문 ID:", order["id"])
    print("주문 상태:", order["status"])
    print("총 주문 금액:", order["total_amount"])
    print("주문 시간:", order["created_at"])

내 주문 개수: 2
------------------------------
주문 ID: f9e1e26f-0d16-4e16-b1b0-1e67044e70c6
주문 상태: pending
총 주문 금액: 20000
주문 시간: 2026-09-13T14:59:25.42164+00:00
------------------------------
주문 ID: 3d939cbc-0290-422b-8e67-18e3280f1371
주문 상태: pending
총 주문 금액: 20000
주문 시간: 2026-09-13T14:59:11.571417+00:00


In [12]:
#cancel_order

order_to_cancel = my_orders[0]

print("취소할 주문 ID:", order_to_cancel["id"])
print("변경 전 상태:", order_to_cancel["status"])

cancelled_order = shop_orders.cancel_order(order_to_cancel["id"])

print("변경 후 상태:", cancelled_order["status"])



취소할 주문 ID: f9e1e26f-0d16-4e16-b1b0-1e67044e70c6
변경 전 상태: pending
변경 후 상태: cancelled


In [31]:

#주문 확인을 위한 판매자 로그인

seller_email = os.getenv("SELLER_EMAIL")
seller_password = os.getenv("SELLER_PASSWORD")

supabase.auth.sign_in_with_password({
    "email": seller_email,
    "password": seller_password
})

current_user = supabase.auth.get_user().user

print("판매자 로그인 완료")
print("현재 사용자 ID:", current_user.id)

판매자 로그인 완료
현재 사용자 ID: f6f9dc8e-d48a-4fec-bc98-09ea3773859f


In [ ]:
#로그인 상태 확인

import importlib
import order_service as osvc

importlib.invalidate_caches()
osvc = importlib.reload(osvc)

shop_orders = osvc.OrderService(supabase)

seller_profile = shop_orders.get_current_profile()

print("현재 역할:", seller_profile["role"])
print(
    "판매자 주문 조회 기능 존재:",
    hasattr(shop_orders, "get_orders_for_my_products")
)

현재 역할: seller
판매자 주문 조회 기능 존재: True


In [15]:
#판매자 주문 내역 조회 

seller_order_items = shop_orders.get_orders_for_my_products()

print("내 상품이 주문된 항목 수:", len(seller_order_items))

for item in seller_order_items:
    order = item["orders"]

    print("-" * 30)
    print("주문 ID:", order["id"])
    print("주문 상태:", order["status"])
    print("상품 ID:", item["product_id"])
    print("주문 수량:", item["quantity"])
    print("주문 당시 개당 가격:", item["unit_price"])

내 상품이 주문된 항목 수: 2
------------------------------
주문 ID: 3d939cbc-0290-422b-8e67-18e3280f1371
주문 상태: pending
상품 ID: 74c19d94-5ece-4be6-a01d-853d391c38fc
주문 수량: 2
주문 당시 개당 가격: 10000
------------------------------
주문 ID: f9e1e26f-0d16-4e16-b1b0-1e67044e70c6
주문 상태: cancelled
상품 ID: 74c19d94-5ece-4be6-a01d-853d391c38fc
주문 수량: 2
주문 당시 개당 가격: 10000


### buyer/seller 권한 테스트

In [17]:
#판매자가 구매자 주문조회 시도

try:
    shop_orders.get_my_orders()

except PermissionError as e:
    print("권한 검사 성공:", e)

권한 검사 성공: 구매자만 주문을 조회할 수 있습니다.


In [39]:
#구매자가 판매자 주문조회 시도

supabase.auth.sign_in_with_password({
    "email": os.getenv("BUYER_EMAIL"),
    "password": os.getenv("BUYER_PASSWORD")
})

current_user = supabase.auth.get_user().user

print("구매자 로그인 완료")
print("현재 사용자 ID:", current_user.id)

구매자 로그인 완료
현재 사용자 ID: 8c9161e5-578e-4000-81fa-560af32eac56


In [19]:
try:
    shop_orders.get_orders_for_my_products()
except PermissionError as e:
    print("권한 검사 성공:", e)

권한 검사 성공: 판매자만 자신의 상품에 대한 주문을 조회할 수 있습니다.


### 원자적 트랜잭션 주문

In [40]:
#commit 테스트

# 김치만두 ID 가져오기
target_product = products[0]

#  [트랜잭션 주문 실행!] 김치만두 2개 주문하기
result = shop_orders.create_order_atomic(
    product_id=target_product["id"],
    quantity=2
)

print(" [COMMIT 성공] 원자적 트랜잭션 주문 완료!")
print(f"- 주문 번호: {result['order_id']}")
print(f"- 상세 품목 번호: {result['order_item_id']}")
print(f"- 주문 수량: {result['quantity']}개")
print(f"- 총 결제 금액: {result['total_amount']}원")
print(f"- 남은 재고: {result['remaining_stock']}개 (100개에서 2개 차감!)")

 [COMMIT 성공] 원자적 트랜잭션 주문 완료!
- 주문 번호: 479f1942-5075-4b6c-8ce3-93a03357ab27
- 상세 품목 번호: 5a1ee8d2-c5b0-4bf3-b793-fe2e378ff666
- 주문 수량: 2개
- 총 결제 금액: 20000원
- 남은 재고: 96개 (100개에서 2개 차감!)


In [25]:
#rollback 테스트

# 💥 [고장 시험] 일부러 에러를 내서 장부가 롤백(ROLLBACK)되는지 시험
rollback_test = shop_orders.test_order_rollback(
    product_id=target_product["id"],
    quantity=1
)

print("🛡️ [ROLLBACK 검증 결과]:")
print(f"- 롤백 성공 여부: {rollback_test.get('rollback_verified')}")
print(f"- 발생한 에러 내용: {rollback_test.get('error_message')}")

🛡️ [ROLLBACK 검증 결과]:
- 롤백 성공 여부: True
- 발생한 에러 내용: {'message': '테스트용 강제 에러! orders 테이블 삽입이 롤백되어야 합니다.', 'code': 'P0001', 'hint': None, 'details': None}


In [24]:
#프로필 수정 태스트

updated = user_service.update_profile(
    address="서울 특별시 마포구 신촌로",
    mobile="010-8788-6573"
)

print("[🎉 Update 성공] 수정된 프로필 정보:")
print(f"새 주소: {updated[0].get('address')}")
print(f"새 휴대폰 번호: {updated[0].get('mobile')}")

[🎉 Update 성공] 수정된 프로필 정보:
새 주소: 서울 특별시 마포구 신촌로
새 휴대폰 번호: 010-8788-6573
